In [ ]:
import numpy as np
import cv2
from skimage.morphology import skeletonize
from scipy.signal import savgol_filter


def validate_criterion_9(
    video_segments: dict,
    frame_start: int,
    frame_end: int,
    object_id: int
):
    """
    Validate Criterion 9: Needle trajectory
    (only insertion count and redressing, no echography criteria)

    needle_masks: list of SAM2 binary masks (H, W)
    """

    masks = []

    for frame_idx in range(frame_start, frame_end + 1):

        if frame_idx not in video_segments:
            continue  # frame absente

        frame_data = video_segments[frame_idx]

        if object_id not in frame_data:
            continue

        masks.append(frame_data[object_id])

    tip_positions = []
    angles = []

    for mask in masks:
        skel = skeleton_from_mask(mask)

        tip = extract_tip_from_skeleton(skel)
        angle = compute_needle_angle(skel)

        if tip is not None:
            tip_positions.append(tip)
        angles.append(angle)

    single_insertion = validate_single_insertion(tip_positions)
    no_redressing = validate_no_redressing(angles)

    criterion_9_valid = single_insertion and no_redressing

    return {
        "criterion_9_valid": criterion_9_valid,
        "single_insertion": single_insertion,
        "no_redressing": no_redressing,
        "n_frames": len(masks)
    }

def skeleton_from_mask(mask: np.ndarray) -> np.ndarray:
    """
    Skeletonize a binary mask (SAM2 output).
    """
    mask_bool = mask.astype(bool)
    skeleton = skeletonize(mask_bool)
    return skeleton.astype(np.uint8)


def extract_tip_from_skeleton(skel: np.ndarray) -> np.ndarray:
    """
    Extract needle tip as the skeleton endpoint farthest from centroid.
    """
    ys, xs = np.where(skel > 0)
    points = np.stack([xs, ys], axis=1)

    if len(points) < 2:
        return None

    centroid = points.mean(axis=0)
    distances = np.linalg.norm(points - centroid, axis=1)
    tip = points[np.argmax(distances)]
    return tip


def compute_needle_angle(skel: np.ndarray) -> float:
    """
    Compute dominant needle angle (in degrees) using PCA on skeleton.
    """
    ys, xs = np.where(skel > 0)
    if len(xs) < 5:
        return None

    points = np.stack([xs, ys], axis=1)
    mean = points.mean(axis=0)
    centered = points - mean

    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    principal_axis = eigvecs[:, np.argmax(eigvals)]

    angle = np.arctan2(principal_axis[1], principal_axis[0])
    return np.degrees(angle)

ModuleNotFoundError: No module named 'skimage'

In [ ]:
def validate_single_insertion(tip_positions, tolerance_px=5):
    """
    Validate that the needle is inserted only once (no withdrawal/re-insertion).

    tip_positions: list of (x, y)
    tolerance_px: tolerated jitter (pixels)
    """
    if len(tip_positions) < 5:
        return False

    tip_positions = np.array(tip_positions)

    # Use y-axis as insertion axis (image coordinates)
    depths = tip_positions[:, 1]

    # Smooth to remove jitter
    depths_smooth = savgol_filter(depths, window_length=5, polyorder=2)

    # First derivative
    velocity = np.diff(depths_smooth)

    # Count sign changes indicating retraction
    retractions = np.sum(velocity < -tolerance_px)

    return retractions == 0

In [ ]:
def validate_no_redressing(angles, max_angle_change_deg=8):
    """
    Validate absence of needle redressing.

    angles: list of angles (degrees)
    max_angle_change_deg: tolerated angular variation
    """
    angles = np.array([a for a in angles if a is not None])

    if len(angles) < 5:
        return False

    # Unwrap angles to avoid 180° jumps
    angles_rad = np.unwrap(np.radians(angles))
    angles_deg = np.degrees(angles_rad)

    angle_variation = np.abs(np.diff(angles_deg))

    return np.max(angle_variation) < max_angle_change_deg

In [ ]:
masks = get_object_masks(h5_path, frame_start, frame_end, object_id )
result = validate_criterion_9(13)

TypeError: 'int' object is not iterable